# 分层索引（Hierarchical Indices）

当文档很长或语料很大时，直接在“细粒度 chunks”上做一次检索，可能会：

- 搜索空间太大（效率低）
- 返回的 chunks 缺少全局上下文（相关但不完整）
- 被局部噪声干扰（相关性不稳定）

分层索引的核心思路是：**先用“粗粒度层”定位大致区域，再在该区域内部做“细粒度层”检索**。

## 1) 整体流程

把同一份内容编码成两种粒度：

- **摘要层（summary store）**：每页/每章一个“短摘要”，用于快速粗筛
- **细节层（detailed store）**：把原文切成 chunks，用于精确命中细节

下面这张图展示了“编码阶段”和“检索阶段”的两层结构。

<div style="text-align: center;">

<img src="../images/hierarchical_indices.svg" alt="hierarchical_indices" style="width:55%; height:auto;">
</div>

<div style="text-align: center;">

<img src="../images/hierarchical_indices_example.svg" alt="hierarchical_indices_example" style="width:100%; height:auto;">
</div>

## 2) 导入依赖

我们会用：

- `pypdf`：按页读取 PDF
- `RecursiveCharacterTextSplitter`：把页内容切成 chunks
- `ChatOpenAI`：生成页级摘要
- `FAISS + OpenAIEmbeddings`：分别构建摘要层/细节层向量库

In [1]:
import os
import sys
import asyncio
import random
from dotenv import load_dotenv
from pathlib import Path
from typing import List, Tuple

import requests
from pypdf import PdfReader

load_dotenv("../.env")

# 让 `all_rag_techniques/` 子目录里的 notebook
# 也能导入上一级目录的 `helper_functions.py`
sys.path.insert(0, str(Path("..").resolve()))

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_50533/1206576930.py:23: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 3) 下载并按页读取 PDF

我们把 PDF **逐页提取**成 `Document` 列表，每个 `Document` 带上 `page` 元数据，后续做“分层检索”会用到它。

In [2]:
os.environ["HTTP_PROXY"] = os.getenv("HTTP_PROXY", "http://127.0.0.1:7890")
os.environ["HTTPS_PROXY"] = os.getenv("HTTPS_PROXY", "http://127.0.0.1:7890")
os.environ["ALL_PROXY"] = os.getenv("ALL_PROXY", "http://127.0.0.1:7890")

In [3]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PDF_URL = "https://raw.githubusercontent.com/NirDiamant/RAG_Techniques/main/data/Understanding_Climate_Change.pdf"
PDF_PATH = DATA_DIR / "Understanding_Climate_Change.pdf"

if not PDF_PATH.exists():
    resp = requests.get(
        PDF_URL,
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    resp.raise_for_status()
    PDF_PATH.write_bytes(resp.content)

reader = PdfReader(str(PDF_PATH))
pages_text = [(p.extract_text() or "") for p in reader.pages]

assert any(t.strip() for t in pages_text), "PDF 提取到的文本为空（pypdf.extract_text 可能失败）"

page_docs: List[Document] = []
for i, txt in enumerate(pages_text):
    t = txt.strip()
    if not t:
        continue
    page_docs.append(Document(page_content=t, metadata={"source": str(PDF_PATH), "page": i}))

print("Pages in PDF:", len(reader.pages))
print("Pages with text:", len(page_docs))
print("Example page (first 300 chars):")
print(page_docs[0].page_content[:300])

Pages in PDF: 33
Pages with text: 33
Example page (first 300 chars):
Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the planet's overall weather patterns, including temperature, 
precipitation, and wind patterns, over an exte


## 4) 生成“页级摘要”（摘要层）

我们用 LLM 把每页压缩成一个短摘要，然后把摘要单独建索引。

为了更快，这里用异步并发；为避免偶发的限流/网络波动，我们加一个指数退避重试。

In [4]:
summary_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

summary_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是文档页级摘要器。给定一页文本，输出一个短摘要，用来支持后续检索粗筛。\n"
            "要求：\n"
            "- 忠实于原文，不要编造\n"
            "- 只输出摘要正文，不要标题/列表编号\n"
            "- 长度控制在 120-180 个中文字符或相近信息量\n",
        ),
        (
            "human",
            "页文本：\n{page_text}\n\n输出（摘要正文）：",
        ),
    ]
)

summary_chain = summary_prompt | summary_llm | StrOutputParser()


async def _retry_summary(payload: dict, max_attempts: int = 6, base_sleep_s: float = 1.0) -> str:
    last_err: Exception | None = None
    for attempt in range(max_attempts):
        try:
            return await summary_chain.ainvoke(payload)
        except Exception as e:
            last_err = e
            if attempt == max_attempts - 1:
                raise
            sleep_s = base_sleep_s * (2**attempt) + random.random() * 0.3
            await asyncio.sleep(sleep_s)
    raise last_err  # pragma: no cover


async def summarize_pages(docs: List[Document], max_concurrency: int = 5) -> List[Document]:
    sem = asyncio.Semaphore(max_concurrency)

    async def _one(d: Document) -> Document:
        async with sem:
            out = (await _retry_summary({"page_text": d.page_content})).strip()
            return Document(
                page_content=out,
                metadata={
                    "source": d.metadata.get("source"),
                    "page": int(d.metadata.get("page", 0)),
                    "summary": True,
                },
            )

    return await asyncio.gather(*[_one(d) for d in docs])


summaries = await summarize_pages(page_docs, max_concurrency=5)
print("Num summaries:", len(summaries))
print("Example summary (first 300 chars):")
print(summaries[0].page_content[:300])

Num summaries: 33
Example summary (first 300 chars):
气候变化指的是全球气候的显著长期变化，主要由人类活动如化石燃料燃烧和森林砍伐引起。历史上，地球气候经历了多次冰川周期，现代气候时代始于约11,700年前。现代科学观察显示，全球温度、海平面和极端天气事件迅速增加，主要由温室气体排放驱动。温室气体如二氧化碳和甲烷通过温室效应保持地球温暖，但人类活动加剧了这一过程，导致气候变暖。


## 5) 构建两级向量库（摘要层 + 细节层）

- **细节层**：把每页拆成 chunks（保留 `page` 元数据）
- **摘要层**：直接用上一节得到的 `summaries`

为了避免每次都重新跑摘要/embedding，我们把两级向量库缓存到本地目录。

In [5]:
chunk_size = 1000
chunk_overlap = 200

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

# 细节层 chunks：由按页 Document 切分，天然带 page 元数据
chunks = splitter.split_documents(page_docs)
for i, c in enumerate(chunks):
    c.metadata.update(
        {
            "chunk_id": i,
            "summary": False,
            "page": int(c.metadata.get("page", 0)),
        }
    )

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

VSTORE_DIR = Path("../vector_stores")
SUMMARY_DIR = VSTORE_DIR / "hierarchical_summary_store"
DETAIL_DIR = VSTORE_DIR / "hierarchical_detailed_store"

if SUMMARY_DIR.is_dir() and DETAIL_DIR.is_dir():
    summary_store = FAISS.load_local(
        str(SUMMARY_DIR),
        embeddings,
        allow_dangerous_deserialization=True,
    )
    detailed_store = FAISS.load_local(
        str(DETAIL_DIR),
        embeddings,
        allow_dangerous_deserialization=True,
    )
else:
    VSTORE_DIR.mkdir(parents=True, exist_ok=True)
    summary_store = FAISS.from_documents(summaries, embeddings)
    detailed_store = FAISS.from_documents(chunks, embeddings)
    summary_store.save_local(str(SUMMARY_DIR))
    detailed_store.save_local(str(DETAIL_DIR))

print("Num detailed chunks indexed:", len(chunks))

Num detailed chunks indexed: 97


## 6) 分层检索：先摘要粗筛，再在目标页里精搜

实现要点：

1. 用 `summary_store` 先找出最相关的若干页（只返回页级摘要）
2. 取出这些页号 `page`，把细节层的候选 chunks 限制在这些页里
3. 在“被限制的候选集”中取出最终 chunks

In [6]:
def retrieve_hierarchical(
    query: str,
    summary_vectorstore: FAISS,
    detailed_vectorstore: FAISS,
    k_summaries: int = 3,
    fetch_k: int = 80,
    k_chunks_per_page: int = 5,
) -> Tuple[List[Document], List[Document]]:
    """两层检索：先页级摘要粗筛，再把候选限制到这些页。"""

    top_summaries = summary_vectorstore.similarity_search(query, k=k_summaries)
    target_pages = [int(s.metadata["page"]) for s in top_summaries]
    target_set = set(target_pages)

    # 细节层先取较多候选，再按 page 做过滤（比依赖向量库的 filter 能力更稳）
    detailed_candidates = detailed_vectorstore.similarity_search(query, k=fetch_k)
    filtered = [d for d in detailed_candidates if int(d.metadata.get("page", -1)) in target_set]

    # 每页最多保留 k_chunks_per_page 个（保持相似度排序）
    per_page_count = {p: 0 for p in target_set}
    out: List[Document] = []
    for d in filtered:
        p = int(d.metadata.get("page", -1))
        if per_page_count.get(p, 0) >= k_chunks_per_page:
            continue
        out.append(d)
        per_page_count[p] = per_page_count.get(p, 0) + 1
        if sum(per_page_count.values()) >= k_chunks_per_page * len(target_set):
            break

    return top_summaries, out


query = "What is the greenhouse effect?"

# Flat 检索：只用细节层
flat_chunks = detailed_store.similarity_search(query, k=8)

# Hierarchical 检索：摘要 -> 目标页 -> 细节
top_summaries, hier_chunks = retrieve_hierarchical(
    query,
    summary_store,
    detailed_store,
    k_summaries=3,
    fetch_k=80,
    k_chunks_per_page=3,
)

print("Query:", query)
print("\nTop summary pages:", [s.metadata["page"] for s in top_summaries])

print("\n=== Flat detailed chunks (top 3) ===")
for i, d in enumerate(flat_chunks[:3], start=1):
    print(f"Chunk {i} (page {d.metadata.get('page')}):")
    print(d.page_content[:350].strip())
    print("---")

print("\n=== Hierarchical chunks (top results) ===")
for i, d in enumerate(hier_chunks, start=1):
    print(f"Chunk {i} (page {d.metadata.get('page')}):")
    print(d.page_content[:350].strip())
    print("---")

Query: What is the greenhouse effect?

Top summary pages: [0, 29, 6]

=== Flat detailed chunks (top 3) ===
Chunk 1 (page 0):
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it
---
Chunk 2 (page 0):
Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. During the Holocene epoch, which 
began at the end of the last ice age, human societies flourished, but the industrial era has seen 
unprecedented changes. 
Modern Observations 
Modern scientific observ
---
Chunk 3 (page 0):
Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes i

## 7) （可选）用同一套 QA 提示词对比两种上下文

把 flat 检索得到的 chunks 和 hierarchical 检索得到的 chunks 分别拼成 `context`，用同一个 QA 提示词问同一个问题，感受“先粗筛再精搜”对答案的影响。

In [7]:
qa_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个严谨的问答助手。只能使用给定的上下文回答问题。\n"
            "如果上下文不包含答案，直接回答：我不知道。",
        ),
        (
            "human",
            "问题：{question}\n\n上下文：\n{context}\n\n回答：",
        ),
    ]
)
qa_chain = qa_prompt | qa_llm | StrOutputParser()


def docs_to_context(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])

flat_context = docs_to_context(flat_chunks)
hier_context = docs_to_context(hier_chunks)

ans_flat = qa_chain.invoke({"question": query, "context": flat_context}).strip()
ans_hier = qa_chain.invoke({"question": query, "context": hier_context}).strip()

print("=== Answer with flat context ===")
print(ans_flat)
print("\n=== Answer with hierarchical context ===")
print(ans_hier)

=== Answer with flat context ===
The greenhouse effect is the process by which greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O), trap heat from the sun in the atmosphere, keeping the planet warm enough to support life. However, human activities have intensified this natural process, leading to a warmer climate.

=== Answer with hierarchical context ===
The greenhouse effect is the process by which greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O), trap heat from the sun in the atmosphere. This effect is essential for life on Earth, as it keeps the planet warm enough to support life. However, human activities have intensified this natural process, leading to a warmer climate.
